<a href="https://colab.research.google.com/github/DerWeiseTeufel/MVM_2025/blob/main/GPU_Training_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata


In [ ]:
import pandas as pd
path = userdata.get('path_to_dataset')


# Предположим, у вас есть DataFrame с товарами
df_products = pd.read_csv(path)
df_products.head()

,NameFull
0,Куртка рабочая сигнальная летняя защитная от о...
1,Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф
2,Анкер химический (капсула с клеевым составом) ...
3,Анкер химический двухкомпонентный Fischer FIS ...
4,Анкер химический Himtex VESF PROFI 200 CAN2004...


In [ ]:
df_products = df_products.rename(columns={'NameFull':'description'})
df_products.head()

,description
0,Куртка рабочая сигнальная летняя защитная от о...
1,Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф
2,Анкер химический (капсула с клеевым составом) ...
3,Анкер химический двухкомпонентный Fischer FIS ...
4,Анкер химический Himtex VESF PROFI 200 CAN2004...


In [ ]:
import pandas as pd
path_test = userdata.get('path_test_rusgidro')


# Предположим, у вас есть DataFrame с товарами
df_test = pd.read_csv(path_test)
df_test.head()

,index,NameFull
0,45983,"Колесо для тачки FIT 77556 16""x4"""
1,16660,Комплект колодок тормозные дисковые передние A...
2,10302,Труба бесшовная горячедеформированная В; с про...
3,8247,Теплообменник кожухотрубчатый горизонтальный Д...
4,3068,Устройство дистанционной защиты Schneider Elec...


In [ ]:
df_test = df_test.rename(columns={'NameFull':'description'})
df_test.head()

,index,description
0,45983,"Колесо для тачки FIT 77556 16""x4"""
1,16660,Комплект колодок тормозные дисковые передние A...
2,10302,Труба бесшовная горячедеформированная В; с про...
3,8247,Теплообменник кожухотрубчатый горизонтальный Д...
4,3068,Устройство дистанционной защиты Schneider Elec...


In [ ]:
COL_NAME = 'description'

# training

In [ ]:
!pip install faiss-cpu # Для GPU-версии

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 90.1 MB/s eta 0:00:00


In [ ]:
import torch
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


pip install torch==2.0.0 transformers==4.30.0 faiss-cpu==1.7.4 pandas==1.5.3 numpy==1.24.0

In [ ]:
!pip install transformers==4.53.3
!pip install trl==0.20.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.1 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 15.0 MB/s eta 0:00:00


In [ ]:
df = df_products
df

,description
0,Куртка рабочая сигнальная летняя защитная от о...
1,Клапан регулирующий черт. 1850973 D75мм 25Х2М1Ф
2,Анкер химический (капсула с клеевым составом) ...
3,Анкер химический двухкомпонентный Fischer FIS ...
4,Анкер химический Himtex VESF PROFI 200 CAN2004...
...,...
49995,Шкаф МКПА комплекса локальной противоаварийной...
49996,Болт M20х150мм шестигранная головка DIN933
49997,Вентиль шаровый DN20 пластиковый
49998,Вентиль шаровый DN50 пластиковый


In [ ]:
from sentence_transformers import SentenceTransformer
sentences = df.description.iloc[:2]

model = SentenceTransformer('sentence-transformers/LaBSE')
embeddings = model.encode(sentences)
print(embeddings)


[[-0.03579483 -0.04024646 -0.00705764 ...  0.03682136 -0.05375298
  -0.01060136]
 [ 0.04000263  0.02426833 -0.01333689 ...  0.02264682 -0.0656014
   0.03192736]]


In [ ]:
model.to(device)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 768, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
  (3): Normalize()
)

In [ ]:
em = []
em.append([[1,2],[3,4]])
em

[[[1, 2], [3, 4]]]

In [ ]:
import numpy as np
em_2 = np.vstack(em)
em_2

array([[1, 2],
       [3, 4]])

In [ ]:
(em_2)

In [ ]:
# create_faiss_index_sbert.py
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from faiss import write_index, IndexFlatIP
import numpy as np



# 1. Load data

texts = df[COL_NAME].tolist()

# 2. Load SBERT model with GPU support
model_name ='sentence-transformers/LaBSE'

print(f"Загрузка модели {model_name}...")

model = SentenceTransformer(model_name)

model.to(device)  # Move model to GPU
model.eval()

# 3. Create embeddings on GPU
embeddings = []
batch_size = 64  # Can increase batch size on GPU

print("Создание эмбеддингов на GPU...")
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encoded = model.encode(batch)
    encoded = np.array(encoded)
    embeddings.append(encoded)

# 4. Save index
embeddings = np.vstack(embeddings)
print(f"Создано эмбеддингов: {embeddings.shape}")

index = IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
write_index(index, "subj_sbert_large_mt_nlu_ru_FAISS.index")

print(f"Индекс SBERT создан: {index.ntotal} векторов")
print(f"Размерность: {embeddings.shape[1]}")

Загрузка модели sentence-transformers/LaBSE...
Создание эмбеддингов на GPU...
Создано эмбеддингов: (50000, 768)
Индекс SBERT создан: 50000 векторов
Размерность: 768


In [ ]:
# import pandas as pd
# import torch
# import torch.nn.functional as F
# from transformers import AutoTokenizer, AutoModel
# from faiss import write_index, IndexFlatIP
# import numpy as np

# # Функция для pooling
# def average_pool(last_hidden_states, attention_mask):
#     last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
#     return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


# # 2. Загрузите модель paraphrase-multilingual-MiniLM-L12-v2
# print("Загрузка модели paraphrase-multilingual-MiniLM-L12-v2...")
# tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
# model = AutoModel.from_pretrained('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
# model.eval()

# # 3. Создайте эмбеддинги для всех текстов
# embeddings = []
# batch_size = 8  # уменьшите если не хватает памяти

# print("Создание эмбеддингов...")
# for i in range(0, len(texts), batch_size):
#     batch = texts[i:i+batch_size]
#     encoded = tokenizer(
#         batch,
#         padding=True,
#         truncation=True,
#         max_length=512,
#         return_tensors='pt'
#     )

#     with torch.no_grad():
#         output = model(**encoded)
#         batch_embeddings = average_pool(
#             output.last_hidden_state,
#             encoded['attention_mask']
#         )
#         batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
#         embeddings.append(batch_embeddings.numpy())

#     if i % 100 == 0:
#         print(f"Обработано {i}/{len(texts)} текстов")

# # 4. Объедините все эмбеддинги
# embeddings = np.vstack(embeddings)
# print(f"Размерность эмбеддингов: {embeddings.shape}")

# # 5. Создайте и сохраните FAISS индекс
# print("Создание FAISS индекса...")
# index = IndexFlatIP(embeddings.shape[1])  # IndexFlatIP для косинусного сходства
# index.add(embeddings)
# write_index(index, "subj_paraphrase_sentences_FAISS.index")

# print(f"Индекс создан и сохранен в subj_paraphrase_sentences_FAISS.index")
# print(f"Количество векторов: {index.ntotal}")

In [ ]:
# create_faiss_index_rubert.py
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from faiss import write_index, IndexFlatIP
import numpy as np

# Функция для pooling
def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


# 1. Load data

texts = df[COL_NAME].tolist()

# 2. Load rubert-tiny2 model with GPU support
print("Загрузка модели  rubert-tiny2...")
tokenizer_rubert_tiny2 = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

model_rubert_tiny2 = AutoModel.from_pretrained("cointegrated/rubert-tiny2")
model_rubert_tiny2.to(device)  # Move model to GPU
model_rubert_tiny2.eval()

# 3. Create embeddings on GPU
embeddings = []
batch_size = 64  # Can increase batch size on GPU

print("Создание эмбеддингов на GPU...")
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encoded = tokenizer_rubert_tiny2(
        batch,
        padding=True,
        truncation=True,
        max_length=24,
        return_tensors='pt'
    )

    # Move tensors to GPU
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        output = model_rubert_tiny2(**encoded)
        batch_embeddings = mean_pooling(output, encoded['attention_mask'])

        # Move embeddings back to CPU for FAISS
        embeddings.append(batch_embeddings.cpu().numpy())

# 4. Save index
embeddings = np.vstack(embeddings)
print(f"Создано эмбеддингов: {embeddings.shape}")

index = IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
write_index(index, "rubert-tiny2_FAISS.index")

print(f"Индекс rubert-tiny2 создан: {index.ntotal} векторов")
print(f"Размерность: {embeddings.shape[1]}")

Загрузка модели  rubert-tiny2...


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Создание эмбеддингов на GPU...
Создано эмбеддингов: (50000, 312)
Индекс rubert-tiny2 создан: 50000 векторов
Размерность: 312


In [ ]:
# create_faiss_index_labse.py
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from faiss import write_index, IndexFlatIP
import numpy as np

# Функция для pooling
def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


# 1. Load data

texts = df[COL_NAME].tolist()

# 2. Load LaBSE model with GPU support
print("Загрузка модели LaBSE...")
tokenizer_labse = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

model_labse = AutoModel.from_pretrained("sentence-transformers/LaBSE")
model_labse.to(device)  # Move model to GPU
model_labse.eval()

# 3. Create embeddings on GPU
embeddings = []
batch_size = 64  # Can increase batch size on GPU

print("Создание эмбеддингов на GPU...")
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encoded = tokenizer_labse(
        batch,
        padding=True,
        truncation=True,
        max_length=24,
        return_tensors='pt'
    )

    # Move tensors to GPU
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        output = model_labse(**encoded)
        batch_embeddings = mean_pooling(output, encoded['attention_mask'])

        # Move embeddings back to CPU for FAISS
        embeddings.append(batch_embeddings.cpu().numpy())

# 4. Save index
embeddings = np.vstack(embeddings)
print(f"Создано эмбеддингов: {embeddings.shape}")

index = IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
write_index(index, "labse_FAISS.index")

print(f"Индекс LaBSE создан: {index.ntotal} векторов")
print(f"Размерность: {embeddings.shape[1]}")

# задачи по обучению на gpu
*  увеличить batch_size
*  

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel
from faiss import read_index
import pandas as pd



# Read the dataset and prepare the subject pack
subj_pack = df[COL_NAME].tolist()
id_pack = df.index.tolist()

# Pydantic model for the response
class SearchResult(BaseModel):
    model_name: str
    k_nearest: List[str]
    k_nearest_ids: List[int]

class SearchResponse(BaseModel):
    results: List[SearchResult]

device = torch.device("cpu")

# Load the tokenizer and model for multilingual-e5-base

index_e5 = read_index("multilingual-e5-base_FAISS.index")
model_e5.to(device)
# Load the tokenizer and model for sbert_large_mt_nlu_ru
tokenizer_sbert = AutoTokenizer.from_pretrained("sberbank-ai/sbert_large_mt_nlu_ru")
model_sbert = model
model_sbert.to(device)

index_sbert = read_index("subj_sbert_large_mt_nlu_ru_FAISS.index")
# Function to perform average pooling on last hidden states
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

def search(text: str = '', k: int = 5):
    # e5
    encoded_input = tokenizer_e5(text, padding=True, truncation=True, max_length=512, return_tensors='pt')
    encoded_input = {key: val.to(device) for key, val in encoded_input.items()}
    with torch.no_grad():
        model_output = model_e5(**encoded_input)
        query_vector = average_pool(model_output.last_hidden_state, encoded_input['attention_mask'])
        query_vector = F.normalize(query_vector, p=2, dim=1)

    # Search in e5 index
    # Convert to numpy on CPU (FAISS requires numpy arrays)
    query_vector_np = query_vector.cpu().numpy().astype('float32')
    D, I = index_e5.search(query_vector_np, k)  # Now it's numpy array
    results = [subj_pack[id] for id in I[0]]
    ids = [id_pack[id] for id in I[0]]
    e5_result = SearchResult(model_name='multilingual-e5-base', k_nearest=results, k_nearest_ids=ids)

    # Process query using sbert model
    encoded_input = tokenizer_sbert(text, padding=True, truncation=True, max_length=24, return_tensors='pt')
    encoded_input = {key: val.to(device) for key, val in encoded_input.items()}
    with torch.no_grad():
        model_output = model_sbert(**encoded_input)
        query_vector = mean_pooling(model_output, encoded_input['attention_mask'])

    # Search in sbert index
    D, I = index_sbert.search(query_vector, k)
    results = [subj_pack[id] for id in I[0]]
    ids = [id_pack[id] for id in I[0]]
    sbert_result = SearchResult(model_name='sbert_large_mt_nlu_ru', k_nearest=results, k_nearest_ids=ids)

    # Process query using paraphrase model
    encoded_input = tokenizer_paraphrase(text, padding=True, truncation=True, max_length=24, return_tensors='pt')
    with torch.no_grad():
        model_output = model_paraphrase(**encoded_input)
        query_vector = mean_pooling(model_output, encoded_input['attention_mask'])

    # Search in paraphrase index
    D, I = index_paraphrase.search(query_vector, k)
    results = [subj_pack[id] for id in I[0]]
    ids = [id_pack[id] for id in I[0]]
    paraphrase_result = SearchResult(model_name='paraphrase-multilingual-MiniLM-L12-v2', k_nearest=results, k_nearest_ids=ids)

    # Return combined results from both models
    paraphrase_result = SearchResult(model_name='paraphrase-multilingual-MiniLM-L12-v2', k_nearest=results, k_nearest_ids=ids)
    return SearchResponse(results=[e5_result, sbert_result, paraphrase_result])


In [ ]:
search("аврора")

NameError: name 'tokenizer_paraphrase' is not defined

In [ ]:
test_df